# 🔧 題目 1：餐飲連鎖 POS 庫存優化
# Mini Data Pipeline 工作坊

> **情境**：你是一家餐飲連鎖集團的資料顧問。老闆想知道客人在抱怨什麼、哪些門店口碑最好。
>
> **你的 pipeline**：
> ```
> CSV → pandas 清洗 → SQLite（raw/cleaned/analyzed 三表）→ SQL 查詢 → LLM 分析 → 報告
> ```
>
> **資料來源**：[Kaggle: 10000 Restaurant Reviews](https://www.kaggle.com/datasets/joebeachcapital/restaurant-reviews)（2,000 筆取樣）

---

## 📋 今日目標

| 必做 | 選做 |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | ⭐ FastAPI API |
| ✅ SQL 查詢統計分析 | ⭐ Streamlit Dashboard |
| ✅ LLM 情緒分析 + 主題分類 | |
| ✅ 顧問報告 output/report.md | |


## Section 0：環境設定

安裝必要套件並設定 API Key。


In [ ]:
# 安裝套件（Colab 已內建 pandas, sqlite3）
# 如果在本地跑，取消下面的註解
# !pip install openai requests

import pandas as pd
import sqlite3
import os
import json

print("✅ 套件載入完成")
print(f"pandas 版本: {pd.__version__}")


In [ ]:
# === API Key 設定 ===
# 方法 1：直接填入（工作坊用，方便但不安全）
# 方法 2：用環境變數（正式專案用）
#
# 💡 沒有 API Key 也能跑！LLM 分析段落有 fallback 規則版。

OPENAI_API_KEY = ""  # 講師會提供共用 Key

# 如果有 .env 檔案
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]

if OPENAI_API_KEY:
    print("✅ API Key 已設定")
else:
    print("⚠️ 沒有 API Key，LLM 分析會使用 fallback 規則版（不影響完成度）")


---
## Section 1：Extract — 讀取資料

> **目標**：把 CSV 讀進來，直接灌入 SQLite 的 `raw_reviews` 表。
> 這是 pipeline 的第一步：**資料進入系統**。

### 為什麼要寫入資料庫？
- 真實 DE 工作中，資料不會一直是 CSV
- 寫入資料庫後，可以用 SQL 查詢、可以被 API 存取、可以被多人共用
- 今天用 SQLite（零安裝），後續升級到 MySQL / BigQuery 概念一樣


In [ ]:
# === 讀取 CSV ===
df_raw = pd.read_csv("data/raw/topic_1/reviews.csv")

print(f"📊 資料筆數: {len(df_raw)}")
print(f"📊 欄位數: {len(df_raw.columns)}")
print(f"📊 欄位: {list(df_raw.columns)}")
print()
print("前 5 筆：")
df_raw.head()


In [ ]:
# === 🔍 檢查資料品質 ===
# 這步很重要：先看資料長什麼樣，再決定怎麼清

print("=== 資料型別 ===")
print(df_raw.dtypes)
print()
print("=== 缺漏值 ===")
print(df_raw.isnull().sum())
print()
print("=== Rating 分佈 ===")
print(df_raw['Rating'].value_counts().sort_index())
print()
print("=== Review 文字長度統計 ===")
print(df_raw['Review'].str.len().describe())


### 🔍 你應該會看到：
- 2,000 筆資料、5 個欄位
- Rating 是 1-5 的整數
- Review 是英文評論文字，長度不一
- 可能有一些缺漏值

**常見問題**：
- 如果 `df_raw` 是空的 → 檢查檔案路徑
- 如果欄位名稱有亂碼 → 可能是編碼問題，試 `encoding='utf-8'` 或 `encoding='latin1'`


In [ ]:
# === 建立 SQLite 資料庫 + 寫入 raw 表 ===
DB_PATH = "pipeline.db"
conn = sqlite3.connect(DB_PATH)

# 把原始資料直接灌入 raw_reviews 表
df_raw.to_sql("raw_reviews", conn, if_exists="replace", index=False)

# ✅ 驗證：用 SQL 查詢確認資料進去了
result = pd.read_sql("SELECT COUNT(*) as total FROM raw_reviews", conn)
print(f"✅ raw_reviews 表已建立，共 {result['total'][0]} 筆")

# 看前 3 筆
pd.read_sql("SELECT * FROM raw_reviews LIMIT 3", conn)


---
## Section 2：Transform — 清洗轉換

> **目標**：從 `raw_reviews` 表讀出資料，清洗後寫入 `cleaned_reviews` 表。
>
> 清洗策略：
> 1. 刪除 Review 或 Rating 為空的列
> 2. Rating 轉成整數
> 3. Time 轉成 datetime，提取年、月
> 4. 加入 review_length 欄位（文字長度）
> 5. 標記低評分（Rating ≤ 2）


In [ ]:
# === 從 raw 表讀出資料 ===
# 注意：從資料庫讀，不是從 CSV！這就是 pipeline 的概念。
df = pd.read_sql("SELECT * FROM raw_reviews", conn)
print(f"從 raw_reviews 讀出 {len(df)} 筆")


### 清洗步驟 1：處理缺漏值

> 🔍 **檢查**：哪些欄位有空值？
> 🎯 **要做**：刪除 Review 或 Rating 為空的列
> 💡 **為什麼**：沒有評論文字就無法做 LLM 分析，沒有評分就無法做統計


In [ ]:
# 清洗前筆數
before = len(df)

# 刪除 Review 或 Rating 為空的列
df = df.dropna(subset=["Review", "Rating"])

# 刪除 Review 太短的（小於 10 字可能是無意義的）
df = df[df["Review"].str.len() >= 10]

print(f"清洗前: {before} 筆 → 清洗後: {len(df)} 筆（刪除 {before - len(df)} 筆）")
print(f"缺漏值: {df.isnull().sum().sum()} 個")


### 清洗步驟 2：型別轉換 + 新增欄位

> 🔍 **檢查**：Rating 是什麼型別？Time 能不能轉日期？
> 🎯 **要做**：轉型別、提取時間特徵、計算文字長度
> 💡 **為什麼**：後續統計需要正確的型別才能做 groupby 和排序


In [ ]:
# Rating 轉整數
df["Rating"] = df["Rating"].astype(int)

# Time 轉 datetime（如果格式允許）
df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
df["year"] = df["Time"].dt.year
df["month"] = df["Time"].dt.month

# 新增欄位
df["review_length"] = df["Review"].str.len()
df["is_negative"] = (df["Rating"] <= 2).astype(int)

print("✅ 型別轉換完成")
print(df.dtypes)
print()
print(f"Rating 範圍: {df['Rating'].min()} - {df['Rating'].max()}")
print(f"負評佔比: {df['is_negative'].mean():.1%}")


### 🏁 清洗檢查點

> 到這裡，確認以下都通過再往下：
> - ✅ 沒有缺漏值
> - ✅ Rating 是 int，範圍 1-5
> - ✅ review_length 和 is_negative 欄位存在


In [ ]:
# === 🏁 清洗檢查點 ===
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值！"
assert df["Rating"].between(1, 5).all(), "❌ Rating 超出 1-5 範圍！"
assert "review_length" in df.columns, "❌ 缺少 review_length 欄位！"
assert "is_negative" in df.columns, "❌ 缺少 is_negative 欄位！"

print("✅ 全部檢查通過！")
print(f"清洗後資料: {len(df)} 筆, {len(df.columns)} 欄")


In [ ]:
# === 寫入 cleaned 表 ===
df.to_sql("cleaned_reviews", conn, if_exists="replace", index=False)

# ✅ 驗證
result = pd.read_sql("SELECT COUNT(*) as total FROM cleaned_reviews", conn)
print(f"✅ cleaned_reviews 表已建立，共 {result['total'][0]} 筆")

# 用 SQL 看三表狀態
print()
for table in ["raw_reviews", "cleaned_reviews"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn)["n"][0]
    print(f"  {table}: {count} 筆")


---
## Section 3：SQL 統計分析

> **目標**：用 SQL 從 `cleaned_reviews` 表做統計分析。
> 
> 要回答的問題：
> 1. 哪些餐廳評分最低？
> 2. 負評佔比多少？各餐廳差異大嗎？
> 3. 有沒有時間趨勢？

> 💡 **為什麼用 SQL 不用 pandas？**
> 真實工作中，資料在資料庫裡，分析師是用 SQL 查詢，不是下載 CSV 再用 pandas。
> 今天兩個都練，但重點是體驗「**從資料庫查詢**」這個動作。


In [ ]:
# === 各餐廳平均評分（SQL 版）===
query = """
SELECT 
    Restaurant,
    COUNT(*) as review_count,
    ROUND(AVG(Rating), 2) as avg_rating,
    ROUND(AVG(is_negative) * 100, 1) as negative_pct
FROM cleaned_reviews
GROUP BY Restaurant
HAVING review_count >= 5
ORDER BY avg_rating ASC
LIMIT 15
"""

restaurant_stats = pd.read_sql(query, conn)
print("📊 評分最低的 15 家餐廳：")
restaurant_stats


In [ ]:
# === 簡單視覺化 ===
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
restaurant_stats.head(10).plot.barh(
    x="Restaurant", y="avg_rating", ax=ax, color="salmon"
)
ax.set_xlabel("平均評分")
ax.set_title("評分最低的 10 家餐廳")
plt.tight_layout()
plt.show()


In [ ]:
# === 負評趨勢（按月）===
query_trend = """
SELECT 
    year, month,
    COUNT(*) as total,
    SUM(is_negative) as negative_count,
    ROUND(AVG(is_negative) * 100, 1) as negative_pct
FROM cleaned_reviews
WHERE year IS NOT NULL
GROUP BY year, month
ORDER BY year, month
"""

trend = pd.read_sql(query_trend, conn)
print("📊 負評趨勢：")
trend


In [ ]:
# === 存統計結果 ===
os.makedirs("data/processed", exist_ok=True)
restaurant_stats.to_csv("data/processed/restaurant_stats.csv", index=False)
print("✅ 統計結果已存到 data/processed/restaurant_stats.csv")


---
## Section 4：LLM 加值分析

> **目標**：用 LLM 對評論文字做情緒分析和主題分類，結果寫入 `analyzed_reviews` 表。
>
> pipeline 位置：
> ```
> raw → cleaned → 【analyzed】← 你在這
> ```
>
> 有 API Key → 呼叫 OpenAI API
> 沒有 API Key → 使用 fallback 規則版（用關鍵字判斷，也能完成）


In [ ]:
# === LLM Helper 函式 ===
import requests

def llm_analyze(text, api_key=None):
    """
    對一段評論文字做情緒分析 + 主題分類。
    有 API Key → 呼叫 OpenAI；沒有 → 使用 fallback 規則。
    回傳 dict: {"sentiment": "...", "topic": "...", "summary": "..."}
    """
    if api_key:
        return _llm_api(text, api_key)
    else:
        return _llm_fallback(text)


def _llm_api(text, api_key):
    """呼叫 OpenAI API"""
    prompt = f"""請分析以下餐廳評論，回傳 JSON 格式：
{{
  "sentiment": "正面/負面/中性",
  "topic": "服務/餐點/價格/環境/其他",
  "summary": "一句話摘要"
}}

評論：{text[:500]}"""
    
    try:
        resp = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "model": "gpt-4o-mini",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.3
            },
            timeout=30
        )
        content = resp.json()["choices"][0]["message"]["content"]
        # 嘗試解析 JSON
        content = content.strip()
        if content.startswith("```"):
            content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except Exception as e:
        print(f"  ⚠️ API 錯誤: {e}，使用 fallback")
        return _llm_fallback(text)


def _llm_fallback(text):
    """規則版 fallback（不需 API Key）"""
    text_lower = text.lower()
    
    # 情緒判斷
    pos_words = ["great", "excellent", "amazing", "love", "best", "good", "delicious", "wonderful", "fantastic"]
    neg_words = ["bad", "worst", "terrible", "awful", "poor", "disgusting", "horrible", "never", "disappointed"]
    
    pos_count = sum(1 for w in pos_words if w in text_lower)
    neg_count = sum(1 for w in neg_words if w in text_lower)
    
    if pos_count > neg_count:
        sentiment = "正面"
    elif neg_count > pos_count:
        sentiment = "負面"
    else:
        sentiment = "中性"
    
    # 主題判斷
    if any(w in text_lower for w in ["service", "staff", "waiter", "rude", "friendly", "slow"]):
        topic = "服務"
    elif any(w in text_lower for w in ["food", "taste", "dish", "menu", "delicious", "cook"]):
        topic = "餐點"
    elif any(w in text_lower for w in ["price", "expensive", "cheap", "cost", "value", "worth"]):
        topic = "價格"
    elif any(w in text_lower for w in ["ambiance", "atmosphere", "decor", "clean", "noise", "view"]):
        topic = "環境"
    else:
        topic = "其他"
    
    return {
        "sentiment": sentiment,
        "topic": topic,
        "summary": text[:50] + "..."
    }

print("✅ LLM Helper 函式已定義")


### ⚡ 先測試 1 筆

> 🔍 **先跑 1 筆看結果合不合理**，再跑批次。
> 💡 prompt 不滿意可以改上面 `_llm_api` 裡的 prompt 再重跑。


In [ ]:
# === 單筆測試 ===
test_text = df["Review"].iloc[0]
print(f"📝 輸入: {test_text[:100]}...")
print()

result = llm_analyze(test_text, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"🤖 分析結果:")
print(f"  情緒: {result['sentiment']}")
print(f"  主題: {result['topic']}")
print(f"  摘要: {result['summary']}")
print()
print("💡 結果合理嗎？不滿意就改上面的 prompt 或 fallback 規則，再重跑這格。")


### 📦 批次分析

> 先跑 **前 50 筆** 確認品質，再決定要不要跑更多。
> ⏱ 使用 API：50 筆約 2-3 分鐘。使用 fallback：幾秒鐘。
>
> 💡 如果 API 太慢或有錯誤，改 `USE_API = False` 切換到 fallback。


In [ ]:
# === 批次分析 ===
USE_API = bool(OPENAI_API_KEY)  # 有 Key 就用 API，沒有就用 fallback
BATCH_SIZE = 50  # 先跑 50 筆，確認品質後可以改大

api_key = OPENAI_API_KEY if USE_API else None
print(f"模式: {'API' if USE_API else 'Fallback 規則版'}")
print(f"分析前 {BATCH_SIZE} 筆...")
print()

results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    result = llm_analyze(row["Review"], api_key)
    results.append(result)
    
    if (len(results)) % 10 == 0:
        print(f"  進度: {len(results)}/{BATCH_SIZE}")

print(f"\n✅ 完成 {len(results)} 筆分析")


In [ ]:
# === 整理 LLM 結果 ===
df_analyzed = df.head(BATCH_SIZE).copy()
df_analyzed["sentiment"] = [r["sentiment"] for r in results]
df_analyzed["topic"] = [r["topic"] for r in results]
df_analyzed["llm_summary"] = [r["summary"] for r in results]

print("📊 情緒分佈：")
print(df_analyzed["sentiment"].value_counts())
print()
print("📊 主題分佈：")
print(df_analyzed["topic"].value_counts())


In [ ]:
# === 寫入 analyzed 表 ===
df_analyzed.to_sql("analyzed_reviews", conn, if_exists="replace", index=False)

# ✅ 驗證三表狀態
print("📊 SQLite 三表狀態：")
for table in ["raw_reviews", "cleaned_reviews", "analyzed_reviews"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn)["n"][0]
    cols = pd.read_sql(f"PRAGMA table_info({table})", conn)
    print(f"  {table}: {count} 筆, {len(cols)} 欄")


---
## Section 5：驗證 pipeline — 跨表查詢

> **目標**：確認三張表的資料是一致的，體驗 data lineage 的概念。
>
> ```
> raw_reviews (2000 筆) → cleaned_reviews (~1900 筆) → analyzed_reviews (50 筆)
> 原始資料              清洗後                        LLM 分析後
> ```
>
> 💡 這就是 data warehouse 分層的雛形：raw → staging → mart


In [ ]:
# === 跨表查詢：比較 raw vs cleaned vs analyzed ===
lineage = pd.read_sql("""
SELECT 
    'raw_reviews' as layer, COUNT(*) as rows FROM raw_reviews
UNION ALL
SELECT 
    'cleaned_reviews', COUNT(*) FROM cleaned_reviews
UNION ALL
SELECT 
    'analyzed_reviews', COUNT(*) FROM analyzed_reviews
""", conn)

print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))

print()
print("💡 raw → cleaned 筆數減少 = 清洗掉了髒資料")
print("💡 cleaned → analyzed 筆數減少 = 只分析了前 50 筆（可以改 BATCH_SIZE 跑更多）")


In [ ]:
# === 從 analyzed 表查洞察 ===
insight = pd.read_sql("""
SELECT 
    topic,
    sentiment,
    COUNT(*) as count,
    ROUND(AVG(Rating), 2) as avg_rating
FROM analyzed_reviews
GROUP BY topic, sentiment
ORDER BY count DESC
""", conn)

print("📊 主題 × 情緒 交叉分析：")
insight


---
## Section 6：產出顧問報告

> **目標**：用分析結果生成一份顧問報告，存到 `output/report.md`。


In [ ]:
# === 生成報告 ===
# 收集統計數據
total = pd.read_sql("SELECT COUNT(*) as n FROM cleaned_reviews", conn)["n"][0]
neg_pct = pd.read_sql("SELECT ROUND(AVG(is_negative)*100, 1) as pct FROM cleaned_reviews", conn)["pct"][0]
top_topics = df_analyzed["topic"].value_counts().head(3).to_dict()
top_sentiments = df_analyzed["sentiment"].value_counts().to_dict()

report = f"""# 餐飲連鎖顧客評論分析報告

## 資料概要
- 分析筆數：{total} 筆顧客評論
- 資料來源：Kaggle Restaurant Reviews
- 分析工具：pandas + SQLite + LLM

## 關鍵發現

### 1. 整體評分
- 負評（Rating ≤ 2）佔比：{neg_pct}%

### 2. 評論主題分佈（LLM 分析 {len(df_analyzed)} 筆）
{chr(10).join(f'- {topic}: {count} 筆' for topic, count in top_topics.items())}

### 3. 情緒分佈
{chr(10).join(f'- {sent}: {count} 筆' for sent, count in top_sentiments.items())}

## 建議
1. 優先處理負面評論中「服務」相關的問題
2. 定期追蹤負評佔比趨勢
3. 將本分析自動化，每週產出一次

## Pipeline 架構
```
CSV → pandas 清洗 → SQLite（raw/cleaned/analyzed）→ SQL 查詢 → LLM 分析 → 本報告
```

## 後續升級
- Docker 容器化 → Airflow 排程 → MySQL/BigQuery → GCP 部署
"""

os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f:
    f.write(report)

print("✅ 報告已存到 output/report.md")
print()
print(report)


---
## Section 7：打包確認

> 最終 checklist：確認所有產出都齊全。


In [ ]:
# === 自動檢查所有產出 ===
checks = [
    ("pipeline.db", "SQLite 資料庫（含 raw/cleaned/analyzed 三表）"),
    ("data/processed/restaurant_stats.csv", "餐廳統計 CSV"),
    ("output/report.md", "顧問分析報告"),
]

print("📋 產出確認：")
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"  {status} {desc}: {path}")
    if not exists:
        all_ok = False

# 檢查 SQLite 三表
if os.path.exists("pipeline.db"):
    conn_check = sqlite3.connect("pipeline.db")
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn_check)
    for t in ["raw_reviews", "cleaned_reviews", "analyzed_reviews"]:
        exists = t in tables["name"].values
        status = "✅" if exists else "❌"
        print(f"  {status} SQLite 表: {t}")
        if not exists:
            all_ok = False
    conn_check.close()

print()
if all_ok:
    print("🎉 全部完成！你的 Mini Data Pipeline 已就緒。")
else:
    print("⚠️ 有缺漏的項目，請回去補完。")

print()
print("📋 接下來：")
print("  1. 填寫 README.md（用講師提供的模板）")
print("  2. 填寫 docs/upgrade_plan.md")
print("  3. 準備 3 分鐘 Demo")
print("  4. （選做）跑 api.py + app.py 看 Dashboard")
